<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания №6


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Movie в C#, который будет представлять информацию о
фильмах. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;
public delegate void MovieReleasedHandler(Movie movie);
public delegate void AwardReceivedHandler(string movieTitle, string awardName);
public delegate void RatingChangedHandler(string movieTitle, double oldRating, double newRating);
public class MovieEventArgs : EventArgs
{
    public string MovieTitle { get; }
    public DateTime EventTime { get; }
    
    public MovieEventArgs(string title)
    {
        MovieTitle = title;
        EventTime = DateTime.Now;
    }
}

public class RatingEventArgs : MovieEventArgs
{
    public double OldRating { get; }
    public double NewRating { get; }
    
    public RatingEventArgs(string title, double oldRating, double newRating) 
        : base(title)
    {
        OldRating = oldRating;
        NewRating = newRating;
    }
}

public interface IAwardWinning
{
    int AwardsCount { get; set; }
    void DisplayAwards();
    event AwardReceivedHandler AwardReceived; 
}

public interface IAwardService
{
    void RegisterAward(string awardName, int year);
    int GetTotalAwards();
    IEnumerable<string> GetAwardHistory();
    event EventHandler<AwardEventArgs> AwardRegistered; 
}

public class AwardEventArgs : EventArgs
{
    public string AwardName { get; }
    public int Year { get; }
    public DateTime RegistrationTime { get; }
    
    public AwardEventArgs(string awardName, int year)
    {
        AwardName = awardName;
        Year = year;
        RegistrationTime = DateTime.Now;
    }
}

public class AwardService : IAwardService
{
    private readonly List<(string Award, int Year)> _awards = new();
    
    public event EventHandler<AwardEventArgs> AwardRegistered;

    public void RegisterAward(string awardName, int year)
    {
        _awards.Add((awardName, year));
        Console.WriteLine($"Зарегистрирована награда: {awardName} ({year} г.)");
        
        AwardRegistered?.Invoke(this, new AwardEventArgs(awardName, year));
    }

    public int GetTotalAwards() => _awards.Count;

    public IEnumerable<string> GetAwardHistory()
    {
        return _awards.Select(a => $"{a.Award} ({a.Year})");
    }
}

public interface IFinancialService
{
    decimal CalculateRevenue(decimal budget, decimal boxOffice);
    decimal CalculateROI(decimal budget, decimal revenue);
    event EventHandler<FinancialEventArgs> FinancialCalculated; 
}

public class FinancialEventArgs : EventArgs
{
    public string MovieTitle { get; }
    public decimal Revenue { get; }
    public decimal ROI { get; }
    
    public FinancialEventArgs(string title, decimal revenue, decimal roi)
    {
        MovieTitle = title;
        Revenue = revenue;
        ROI = roi;
    }
}

public class FinancialService : IFinancialService
{
    public event EventHandler<FinancialEventArgs> FinancialCalculated;
    
    public decimal CalculateRevenue(decimal budget, decimal boxOffice)
    {
        var revenue = boxOffice - budget;
        return revenue;
    }

    public decimal CalculateROI(decimal budget, decimal revenue)
    {
        var roi = budget > 0 ? (revenue / budget) * 100 : 0;
        return roi;
    }
    
    public void CalculateAndNotify(string movieTitle, decimal budget, decimal boxOffice)
    {
        var revenue = CalculateRevenue(budget, boxOffice);
        var roi = CalculateROI(budget, revenue);
        
        FinancialCalculated?.Invoke(this, new FinancialEventArgs(movieTitle, revenue, roi));
    }
}

public class FilmCollection<T> where T : Movie
{
    private List<T> _films = new List<T>();
    
    public event EventHandler<CollectionChangedEventArgs<T>> CollectionChanged;
    
    public void AddFilm(T film)
    {
        _films.Add(film);
        Console.WriteLine($"Фильм «{film.Title}» добавлен в коллекцию");
        
        CollectionChanged?.Invoke(this, 
            new CollectionChangedEventArgs<T>(film, CollectionChangeType.Added));
    }
    
    public void RemoveFilm(T film)
    {
        if (_films.Remove(film))
        {
            Console.WriteLine($"Фильм «{film.Title}» удален из коллекции");
            CollectionChanged?.Invoke(this, 
                new CollectionChangedEventArgs<T>(film, CollectionChangeType.Removed));
        }
    }
    
    public T FindFilmByTitle(string title)
    {
        return _films.FirstOrDefault(f => 
            f.Title.Equals(title, StringComparison.OrdinalIgnoreCase));
    }
    
    public void DisplayAllFilms()
    {
        Console.WriteLine($"\nФильмы в коллекции ({typeof(T).Name}):");
        foreach (var film in _films)
        {
            Console.WriteLine($"- {film.GetInfo()}");
        }
    }
    
    public IEnumerable<T> GetFilmsByDirector(string director)
    {
        return _films.Where(f => 
            f.Director.Equals(director, StringComparison.OrdinalIgnoreCase));
    }
    
    public IEnumerable<T> GetFilmsWithRatingAbove(double minRating)
    {
        return _films.Where(f => f.CalculateRating() >= minRating);
    }
    
    public CollectionStatistics GetStatistics()
    {
        return new CollectionStatistics
        {
            TotalFilms = _films.Count,
            AverageRating = _films.Any() ? _films.Average(f => f.CalculateRating()) : 0,
            AverageYear = _films.Any() ? _films.Average(f => f.Year) : 0,
            OldestFilm = _films.OrderBy(f => f.Year).FirstOrDefault(),
            NewestFilm = _films.OrderByDescending(f => f.Year).FirstOrDefault()
        };
    }
}

public enum CollectionChangeType
{
    Added,
    Removed,
    Updated
}

public class CollectionChangedEventArgs<T> : EventArgs where T : Movie
{
    public T Movie { get; }
    public CollectionChangeType ChangeType { get; }
    
    public CollectionChangedEventArgs(T movie, CollectionChangeType changeType)
    {
        Movie = movie;
        ChangeType = changeType;
    }
}

public class CollectionStatistics
{
    public int TotalFilms { get; set; }
    public double AverageRating { get; set; }
    public double AverageYear { get; set; }
    public Movie OldestFilm { get; set; }
    public Movie NewestFilm { get; set; }
    
    public void Display()
    {
        Console.WriteLine($"Всего фильмов: {TotalFilms}");
        Console.WriteLine($"Средний рейтинг: {AverageRating:F1}");
        Console.WriteLine($"Средний год выпуска: {AverageYear:F0}");
        if (OldestFilm != null)
            Console.WriteLine($"Самый старый фильм: {OldestFilm.Title} ({OldestFilm.Year})");
        if (NewestFilm != null)
            Console.WriteLine($"Самый новый фильм: {NewestFilm.Title} ({NewestFilm.Year})");
    }
}

public class Review
{
    public string Author { get; set; }
    public int Rating { get; set; }
    public string Comment { get; set; }
    public DateTime ReviewDate { get; set; }
    public int HelpfulVotes { get; set; } 
    public int UnhelpfulVotes { get; set; } 
    
    public Review(string author, int rating, string comment)
    {
        Author = author;
        Rating = rating;
        Comment = comment;
        ReviewDate = DateTime.Now;
        HelpfulVotes = 0;
        UnhelpfulVotes = 0;
    }
    
    public Review(string author, int rating) : this(author, rating, "Без комментария") 
    {
    }
    
    public string GetReviewInfo()
    {
        return $"{Author} ({ReviewDate:yyyy-MM-dd}): {Rating}/10 - {Comment} " +
               $"(Полезно: {HelpfulVotes}, Бесполезно: {UnhelpfulVotes})";
    }

    public string GetReviewInfo(bool detailed)
    {
        if (!detailed)
            return $"{Author}: {Rating}/10 (рейтинг полезности: {GetHelpfulnessRatio():P0})";
        
        return GetReviewInfo();
    }

    public bool IsRecent() 
    {
        return (DateTime.Now - ReviewDate).TotalDays <= 30;
    }
    
    public double GetHelpfulnessRatio()
    {
        int totalVotes = HelpfulVotes + UnhelpfulVotes;
        return totalVotes > 0 ? (double)HelpfulVotes / totalVotes : 0;
    }
    
    public void VoteHelpful()
    {
        HelpfulVotes++;
        Console.WriteLine($"Отзыв от {Author} отмечен как полезный");
    }
    
    public void VoteUnhelpful()
    {
        UnhelpfulVotes++;
        Console.WriteLine($"Отзыв от {Author} отмечен как бесполезный");
    }
    
    public bool IsExpertReview()
    {
        return Author.Contains("Критик") || Author.Contains("Critic") || 
               Author.Contains("Эксперт") || Author.Contains("Expert");
    }
}

public class Movie
{
    private string _title;
    private int _year;
    private string _director;
    private List<Review> _reviews;
    private int _duration;
    private string _country; 
    private string _language;
    private string _productionStudio;
    private List<string> _languages;
    private List<string> _actors; 
    private List<string> _tags; 
    private bool _isReleased;
    
    public event MovieReleasedHandler MovieReleased;
    public event RatingChangedHandler RatingChanged;
    public event EventHandler<MovieEventArgs> MovieViewed;
    
    public string Title
    {
        get => _title;
        set => _title = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Название не может быть пустым");
    }

    public int Year
    {
        get => _year;
        set => _year = (value >= 1888 && value <= DateTime.Now.Year + 2) ? value : throw new ArgumentException("Некорректный год выпуска");
    }

    public string Director
    {
        get => _director;
        set => _director = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Имя режиссера не может быть пустым");
    }

    public int Duration
    {
        get => _duration;
        set => _duration = value > 0 ? value : throw new ArgumentException("Продолжительность должна быть положительной");
    }

    public string Country
    {
        get => _country;
        set => _country = !string.IsNullOrWhiteSpace(value) ? value : "Не указана";
    }

    public string Language
    {
        get => _language;
        set => _language = !string.IsNullOrWhiteSpace(value) ? value : "Английский";
    }

    public string ProductionStudio
    {
        get => _productionStudio;
        set => _productionStudio = !string.IsNullOrWhiteSpace(value) ? value : "Неизвестно";
    }

    public bool IsReleased
    {
        get => _isReleased;
        private set => _isReleased = value;
    }

    public IReadOnlyList<string> Languages => _languages.AsReadOnly();
    public IReadOnlyList<string> Actors => _actors.AsReadOnly(); 
    public IReadOnlyList<string> Tags => _tags.AsReadOnly(); 
    public IReadOnlyList<Review> Reviews => _reviews.AsReadOnly();

    public Movie(string title, int year, string director, int duration)
    {
        Title = title;
        Year = year;
        Director = director;
        Duration = duration;
        _reviews = new List<Review>();
        _languages = new List<string>();
        _actors = new List<string>(); 
        _tags = new List<string>(); 
        Country = "Не указана";
        Language = "Английский";
        ProductionStudio = "Неизвестно";
        IsReleased = false;
        
        _languages.Add(Language);
    }

    public Movie(string title, int year, string director, int duration, string country, string language) 
        : this(title, year, director, duration)
    {
        Country = country;
        Language = language;
        if (!_languages.Contains(language))
            _languages.Add(language);
    }

    public Movie(string title, int year, string director, int duration, string country, 
                string language, string productionStudio) 
        : this(title, year, director, duration, country, language)
    {
        ProductionStudio = productionStudio;
    }

    public virtual void Release()
    {
        if (!IsReleased)
        {
            IsReleased = true;
            Console.WriteLine($"Фильм «{Title}» выпущен в {Year} году!");
            MovieReleased?.Invoke(this);
        }
    }

    public void MarkAsViewed()
    {
        Console.WriteLine($"Фильм «{Title}» отмечен как просмотренный");
        MovieViewed?.Invoke(this, new MovieEventArgs(Title));
    }

    public virtual string GetInfo()
    {
        return $"«{Title}» ({Year}), реж. {Director}, {Duration} мин, {Country}";
    }

    public string GetInfo(bool detailed)
    {
        if (!detailed)
            return $"«{Title}» ({Year}), {Director}";
        
        return GetInfo() + $", язык: {Language}";
    }

    public virtual double CalculateRating()
    {
        if (!_reviews.Any()) return 5.0;
        
        double oldRating = _reviews.Any() ? _reviews.Average(r => r.Rating) : 5.0;
        double newRating = _reviews.Average(r => r.Rating);
        
        if (Math.Abs(oldRating - newRating) > 0.1)
        {
            RatingChanged?.Invoke(Title, oldRating, newRating);
        }
        
        return newRating;
    }

    public virtual string GetAgeCategory()
    {
        int currentYear = DateTime.Now.Year;
        int age = currentYear - Year;
        
        return age switch
        {
            < 5 => "Новый",
            < 20 => "Современный",
            < 50 => "Классика",
            _ => "Старая классика"
        };
    }

    public void AddReview(Review review)
    {
        double oldRating = CalculateRating();
        _reviews.Add(review);
        double newRating = CalculateRating();
        
        Console.WriteLine($"Добавлен отзыв для «{Title}» от {review.Author}");
        
        if (Math.Abs(oldRating - newRating) > 0.5)
        {
            RatingChanged?.Invoke(Title, oldRating, newRating);
        }
    }

    public void AddReview(string author, int rating, string comment)
    {
        AddReview(new Review(author, rating, comment));
    }

    public void ShowAllReviews()
    {
        Console.WriteLine($"\nОтзывы для фильма «{Title}»:");
        foreach (var review in _reviews)
        {
            string recent = review.IsRecent() ? " (НОВЫЙ)" : "";
            Console.WriteLine($"- {review.GetReviewInfo()}{recent}");
        }
    }

    public void ShowAllReviews(bool shortVersion)
    {
        if (!shortVersion)
        {
            ShowAllReviews();
            return;
        }
        
        Console.WriteLine($"\nКраткие отзывы для «{Title}»:");
        foreach (var review in _reviews)
        {
            Console.WriteLine($"- {review.GetReviewInfo(false)}");
        }
    }

    public void RecommendSimilar(List<Movie> movies)
    {
        var similar = movies
            .Where(m => m != this && (m.Director == Director || m.Tags.Intersect(Tags).Any()))
            .Take(3)
            .ToList();

        if (similar.Any())
        {
            Console.WriteLine($"\nЕсли вам понравился «{Title}», рекомендуем:");
            foreach (var movie in similar)
            {
                Console.WriteLine($"- {movie.GetInfo()}");
            }
        }
    }

    public string GetDurationCategory() 
    {
        return Duration switch
        {
            < 60 => "Короткометражка",
            < 120 => "Среднеметражка",
            _ => "Полнометражка"
        };
    }

    public bool IsInternational()
    {
        return !string.IsNullOrEmpty(Country) && 
               !Country.Equals("США", StringComparison.OrdinalIgnoreCase) &&
               !Country.Equals("USA", StringComparison.OrdinalIgnoreCase);
    }

    public void AddLanguage(string language)
    {
        if (!string.IsNullOrWhiteSpace(language) && !_languages.Contains(language))
        {
            _languages.Add(language);
            Console.WriteLine($"Добавлен язык: {language} для фильма «{Title}»");
        }
    }

    public virtual string GetTechnicalInfo()
    {
        return $"Студия: {ProductionStudio}, Языки: {string.Join(", ", _languages)}";
    }

    public bool IsMultilingual()
    {
        return _languages.Count > 1;
    }

    public int GetLanguageCount()
    {
        return _languages.Count;
    }
    
    public void AddActor(string actorName)
    {
        if (!string.IsNullOrWhiteSpace(actorName) && !_actors.Contains(actorName))
        {
            _actors.Add(actorName);
            Console.WriteLine($"Добавлен актер: {actorName} в фильм «{Title}»");
        }
    }
    
    public void AddActors(params string[] actors)
    {
        foreach (var actor in actors)
        {
            AddActor(actor);
        }
    }
    
    public bool HasActor(string actorName)
    {
        return _actors.Any(a => a.Equals(actorName, StringComparison.OrdinalIgnoreCase));
    }
    
    public void DisplayCast()
    {
        if (_actors.Any())
        {
            Console.WriteLine($"\nАктерский состав «{Title}»:");
            foreach (var actor in _actors)
            {
                Console.WriteLine($"- {actor}");
            }
        }
    }
    
    public void AddTag(string tag)
    {
        if (!string.IsNullOrWhiteSpace(tag) && !_tags.Contains(tag))
        {
            _tags.Add(tag);
            Console.WriteLine($"Добавлен тег: {tag} к фильму «{Title}»");
        }
    }
    
    public void AddTags(params string[] tags)
    {
        foreach (var tag in tags)
        {
            AddTag(tag);
        }
    }
    
    public bool HasTag(string tag)
    {
        return _tags.Contains(tag, StringComparer.OrdinalIgnoreCase);
    }
    
    public IEnumerable<Movie> FindSimilarByTags(List<Movie> allMovies, int minCommonTags = 2)
    {
        return allMovies
            .Where(m => m != this)
            .Where(m => m.Tags.Intersect(Tags, StringComparer.OrdinalIgnoreCase).Count() >= minCommonTags)
            .OrderByDescending(m => m.Tags.Intersect(Tags, StringComparer.OrdinalIgnoreCase).Count());
    }
    
    public ReviewAnalysis AnalyzeReviews()
    {
        if (!_reviews.Any())
            return new ReviewAnalysis();
        
        return new ReviewAnalysis
        {
            TotalReviews = _reviews.Count,
            AverageRating = CalculateRating(),
            ExpertReviewsCount = _reviews.Count(r => r.IsExpertReview()),
            RecentReviewsCount = _reviews.Count(r => r.IsRecent()),
            MostHelpfulReview = _reviews.OrderByDescending(r => r.GetHelpfulnessRatio()).FirstOrDefault()
        };
    }
}

public class ReviewAnalysis
{
    public int TotalReviews { get; set; }
    public double AverageRating { get; set; }
    public int ExpertReviewsCount { get; set; }
    public int RecentReviewsCount { get; set; }
    public Review MostHelpfulReview { get; set; }
    
    public void Display()
    {
        Console.WriteLine($"Всего отзывов: {TotalReviews}");
        Console.WriteLine($"Средний рейтинг: {AverageRating:F1}");
        Console.WriteLine($"Экспертных отзывов: {ExpertReviewsCount}");
        Console.WriteLine($"Новых отзывов (за 30 дней): {RecentReviewsCount}");
        if (MostHelpfulReview != null)
        {
            Console.WriteLine($"Самый полезный отзыв: {MostHelpfulReview.Author} " +
                             $"(рейтинг полезности: {MostHelpfulReview.GetHelpfulnessRatio():P0})");
        }
    }
}

public class FeatureFilm : Movie, IAwardWinning
{
    private string _genre;
    private double _budget;
    private string _productionCompany;
    private decimal _boxOffice;
    private readonly IFinancialService _financialService;
    private string _mainActor; 
    private List<string> _screenwriters; 
    
    public event EventHandler<BoxOfficeEventArgs> BoxOfficeMilestoneReached;

    public int AwardsCount { get; set; }
    
    public event AwardReceivedHandler AwardReceived;

    public string Genre
    {
        get => _genre;
        set => _genre = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Жанр не может быть пустым");
    }

    public double Budget
    {
        get => _budget;
        set => _budget = value >= 0 ? value : throw new ArgumentException("Бюджет не может быть отрицательным");
    }

    public string ProductionCompany
    {
        get => _productionCompany;
        set => _productionCompany = !string.IsNullOrWhiteSpace(value) ? value : "Неизвестно";
    }

    public decimal BoxOffice
    {
        get => _boxOffice;
        set
        {
            decimal oldValue = _boxOffice;
            _boxOffice = value >= 0 ? value : 0;
            
            CheckBoxOfficeMilestones(oldValue, value);
        }
    }
    
    public string MainActor
    {
        get => _mainActor;
        set
        {
            if (!string.IsNullOrWhiteSpace(value))
            {
                _mainActor = value;
                AddActor(value); 
            }
        }
    }
    
    public IReadOnlyList<string> Screenwriters => _screenwriters.AsReadOnly();

    public FeatureFilm(string title, int year, string director, string genre, double budget, int duration) 
        : base(title, year, director, duration)
    {
        Genre = genre;
        Budget = budget;
        AwardsCount = 0;
        ProductionCompany = "Неизвестно";
        _financialService = new FinancialService();
        _screenwriters = new List<string>(); 
        SubscribeToFinancialEvents();
    }

    public FeatureFilm(string title, int year, string director, string genre, double budget, int duration, 
        string country, string language, string productionCompany) 
        : base(title, year, director, duration, country, language)
    {
        Genre = genre;
        Budget = budget;
        ProductionCompany = productionCompany;
        AwardsCount = 0;
        _financialService = new FinancialService();
        _screenwriters = new List<string>();
        SubscribeToFinancialEvents();
    }

    public FeatureFilm(string title, int year, string director, string genre, 
                      double budget, int duration, IFinancialService financialService)
        : base(title, year, director, duration)
    {
        Genre = genre;
        Budget = budget;
        _financialService = financialService ?? new FinancialService();
        AwardsCount = 0;
        ProductionCompany = "Неизвестно";
        _screenwriters = new List<string>();
        SubscribeToFinancialEvents();
    }

    public FeatureFilm(string title, int year, string director, string genre, double budget, int duration,
        string country, string language, string productionCompany, IFinancialService financialService)
        : base(title, year, director, duration, country, language, productionCompany)
    {
        Genre = genre;
        Budget = budget;
        ProductionCompany = productionCompany;
        _financialService = financialService ?? new FinancialService();
        AwardsCount = 0;
        _screenwriters = new List<string>();
        SubscribeToFinancialEvents();
    }
    
    private void SubscribeToFinancialEvents()
    {
        if (_financialService is FinancialService fs)
        {
            fs.FinancialCalculated += (sender, args) =>
            {
                Console.WriteLine($"Финансовый расчет для «{Title}»: " +
                                $"Выручка: ${args.Revenue:F1} млн, ROI: {args.ROI:F1}%");
            };
        }
    }
    
    private void CheckBoxOfficeMilestones(decimal oldValue, decimal newValue)
    {
        decimal[] milestones = { 100, 200, 300, 400, 500, 1000 };
        
        foreach (var milestone in milestones)
        {
            if (oldValue < milestone && newValue >= milestone)
            {
                BoxOfficeMilestoneReached?.Invoke(this, 
                    new BoxOfficeEventArgs(Title, milestone, newValue));
            }
        }
    }

    public override string GetInfo()
    {
        string mainActorInfo = !string.IsNullOrEmpty(MainActor) ? $", главный актер: {MainActor}" : "";
        return base.GetInfo() + $", жанр: {Genre}, бюджет: ${Budget} млн{mainActorInfo}";
    }

    public string GetInfo(bool includeCompany)
    {
        string info = GetInfo();
        if (includeCompany)
            info += $", студия: {ProductionCompany}";
        return info;
    }

    public override double CalculateRating()
    {
        double rating = base.CalculateRating();
        if (Budget > 100) rating += 1.0;
        if (AwardsCount > 0) rating += AwardsCount * 0.1;
        return Math.Min(rating, 10.0);
    }

    public override string GetAgeCategory()
    {
        string baseCategory = base.GetAgeCategory();
        if (Budget > 200 && baseCategory == "Новый")
            return "Блокбастер";
        return baseCategory;
    }

    public void ShowTrailer()
    {
        Console.WriteLine($"Просмотр трейлера фильма «{Title}»");
    }

    public void ShowTrailer(string platform)
    {
        Console.WriteLine($"Просмотр трейлера фильма «{Title}» на {platform}");
    }

    public void AddToFilmFestival(List<FeatureFilm> festivalFilms)
    {
        festivalFilms.Add(this);
        Console.WriteLine($"Фильм «{Title}» добавлен в кинофестиваль!");
    }

    public void DisplayAwards() 
    {
        Console.WriteLine($"Фильм «{Title}» получил {AwardsCount} наград");
    }
    
    public void ReceiveAward(string awardName)
    {
        AwardsCount++;
        Console.WriteLine($"Фильм «{Title}» получил награду: {awardName}");
        AwardReceived?.Invoke(Title, awardName);
    }

    public bool IsBlockbuster()
    {
        return Budget > 100 && CalculateRating() >= 7.0;
    }

    public decimal CalculateRevenue()
    {
        return _financialService.CalculateRevenue((decimal)Budget, BoxOffice);
    }

    public decimal CalculateROI()
    {
        var revenue = CalculateRevenue();
        return _financialService.CalculateROI((decimal)Budget, revenue);
    }

    public string GetFinancialReport()
    {
        var revenue = CalculateRevenue();
        var roi = CalculateROI();
        var profitStatus = revenue >= 0 ? "Прибыль" : "Убыток";
        
        return $"Финансовый отчет «{Title}»:\n" +
               $"  Бюджет: ${Budget} млн\n" +
               $"  Сборы: ${BoxOffice} млн\n" +
               $"  {profitStatus}: ${Math.Abs(revenue):F1} млн\n" +
               $"  ROI: {roi:F1}%";
    }

    public override string GetTechnicalInfo()
    {
        string writersInfo = _screenwriters.Any() ? $", сценаристы: {string.Join(", ", _screenwriters)}" : "";
        return base.GetTechnicalInfo() + $", сборы: ${BoxOffice} млн{writersInfo}";
    }
    
    public void AddScreenwriter(string screenwriter)
    {
        if (!string.IsNullOrWhiteSpace(screenwriter) && !_screenwriters.Contains(screenwriter))
        {
            _screenwriters.Add(screenwriter);
            Console.WriteLine($"Добавлен сценарист: {screenwriter} для фильма «{Title}»");
        }
    }
    
    public bool IsFranchiseFilm()
    {
        return Title.Contains(":") || Title.Contains("Часть") || 
               Title.Contains("Part") || Title.Contains("серия", StringComparison.OrdinalIgnoreCase);
    }
    
    public decimal ForecastFutureRevenue(double growthRate = 0.1)
    {
        return BoxOffice * (decimal)(1 + growthRate);
    }
}

public class BoxOfficeEventArgs : EventArgs
{
    public string MovieTitle { get; }
    public decimal Milestone { get; }
    public decimal CurrentBoxOffice { get; }
    
    public BoxOfficeEventArgs(string title, decimal milestone, decimal currentBoxOffice)
    {
        MovieTitle = title;
        Milestone = milestone;
        CurrentBoxOffice = currentBoxOffice;
    }
}

public class Documentary : Movie, IAwardWinning
{
    private string _topic;
    private bool _isEducational;
    private string _researchInstitution;
    private List<string> _experts; 
    private string _narrator; 
    
    public int AwardsCount { get; set; }
    public event AwardReceivedHandler AwardReceived;

    public string Topic
    {
        get => _topic;
        set => _topic = !string.IsNullOrWhiteSpace(value) ? value : throw new ArgumentException("Тема не может быть пустой");
    }

    public bool IsEducational
    {
        get => _isEducational;
        set => _isEducational = value;
    }

    public string ResearchInstitution
    {
        get => _researchInstitution;
        set => _researchInstitution = !string.IsNullOrWhiteSpace(value) ? value : "Не указано";
    }
    
    public string Narrator
    {
        get => _narrator;
        set
        {
            if (!string.IsNullOrWhiteSpace(value))
            {
                _narrator = value;
                AddActor(value); 
            }
        }
    }
    
    public IReadOnlyList<string> Experts => _experts.AsReadOnly();

    public Documentary(string title, int year, string director, string topic, bool isEducational, int duration) 
        : base(title, year, director, duration)
    {
        Topic = topic;
        IsEducational = isEducational;
        AwardsCount = 0;
        ResearchInstitution = "Не указано";
        _experts = new List<string>(); 
    }

    public Documentary(string title, int year, string director, string topic, bool isEducational, int duration,
        string country, string language, string researchInstitution) 
        : base(title, year, director, duration, country, language)
    {
        Topic = topic;
        IsEducational = isEducational;
        ResearchInstitution = researchInstitution;
        AwardsCount = 0;
        _experts = new List<string>();
    }

    public override string GetInfo()
    {
        string narratorInfo = !string.IsNullOrEmpty(Narrator) ? $", рассказчик: {Narrator}" : "";
        string edu = IsEducational ? "образовательный" : "популярный";
        return base.GetInfo() + $", тема: {Topic} ({edu}){narratorInfo}";
    }

    public string GetInfo(bool includeInstitution)
    {
        string info = GetInfo();
        if (includeInstitution && !string.IsNullOrEmpty(ResearchInstitution))
            info += $", институт: {ResearchInstitution}";
        return info;
    }

    public override double CalculateRating()
    {
        double rating = base.CalculateRating();
        if (IsEducational) rating += 0.5;
        if (AwardsCount > 0) rating += AwardsCount * 0.2;
        return Math.Min(rating, 10.0);
    }

    public override string GetAgeCategory()
    {
        int currentYear = DateTime.Now.Year;
        int age = currentYear - Year;
        
        if (IsEducational && age <= 10)
            return "Актуальное исследование";
        
        return base.GetAgeCategory();
    }

    public void ConductInterview()
    {
        Console.WriteLine($"Проведение интервью по теме: {Topic}");
    }

    public void ConductInterview(string interviewee)
    {
        Console.WriteLine($"Проведение интервью с {interviewee} по теме: {Topic}");
    }

    public void AddToScienceConference(List<Documentary> scienceFilms)
    {
        if (IsEducational)
        {
            scienceFilms.Add(this);
            Console.WriteLine($"Документальный фильм «{Title}» добавлен в научную конференцию!");
        }
    }

    public void DisplayAwards() 
    {
        string type = IsEducational ? "Научный" : "Документальный";
        Console.WriteLine($"{type} фильм «{Title}» получил {AwardsCount} наград");
    }
    
    public void ReceiveAward(string awardName)
    {
        AwardsCount++;
        Console.WriteLine($"Документальный фильм «{Title}» получил награду: {awardName}");
        AwardReceived?.Invoke(Title, awardName);
    }

    public bool HasAcademicSupport()
    {
        return IsEducational && !string.IsNullOrEmpty(ResearchInstitution) && 
               !ResearchInstitution.Equals("Не указано");
    }

    public string GetResearchStatus()
    {
        if (!IsEducational) return "Популярный фильм";
        
        return HasAcademicSupport() 
            ? $"Научное исследование при поддержке {ResearchInstitution}"
            : "Независимое исследование";
    }
    
    public void AddExpert(string expertName)
    {
        if (!string.IsNullOrWhiteSpace(expertName) && !_experts.Contains(expertName))
        {
            _experts.Add(expertName);
            Console.WriteLine($"Добавлен эксперт: {expertName} для фильма «{Title}»");
        }
    }
    
    public IEnumerable<string> GetAllParticipants()
    {
        var participants = new List<string>();
        if (!string.IsNullOrEmpty(Director)) participants.Add($"Режиссер: {Director}");
        if (!string.IsNullOrEmpty(Narrator)) participants.Add($"Рассказчик: {Narrator}");
        participants.AddRange(_experts.Select(e => $"Эксперт: {e}"));
        participants.AddRange(Actors.Select(a => $"Актер: {a}"));
        
        return participants.Distinct();
    }
    
    public bool IsScientificallyAccurate()
    {
        return IsEducational && _experts.Count >= 2;
    }
}

public class HybridFilm : FeatureFilm, IAwardWinning
{
    private bool _hasDocumentaryElements;
    private int _archiveFootageMinutes;
    private List<string> _historicalConsultants; 
    public new int AwardsCount { get; set; }
    public new event AwardReceivedHandler AwardReceived;

    public HybridFilm(string title, int year, string director, string genre, 
        double budget, int duration, bool hasDocElements) 
        : base(title, year, director, genre, budget, duration)
    {
        _hasDocumentaryElements = hasDocElements;
        _archiveFootageMinutes = 0;
        _historicalConsultants = new List<string>(); 
    }

    public HybridFilm(string title, int year, string director, string genre, 
        double budget, int duration, bool hasDocElements, string country, 
        string language, string productionCompany, int archiveFootage) 
        : base(title, year, director, genre, budget, duration, country, language, productionCompany)
    {
        _hasDocumentaryElements = hasDocElements;
        _archiveFootageMinutes = archiveFootage;
        _historicalConsultants = new List<string>();
    }

    public int ArchiveFootageMinutes
    {
        get => _archiveFootageMinutes;
        set => _archiveFootageMinutes = value >= 0 ? value : 0;
    }
    
    public IReadOnlyList<string> HistoricalConsultants => _historicalConsultants.AsReadOnly();

    public override string GetInfo()
    {
        string hybrid = _hasDocumentaryElements ? " (гибридный)" : "";
        string footage = _archiveFootageMinutes > 0 ? $", архивные кадры: {_archiveFootageMinutes} мин" : "";
        string consultants = _historicalConsultants.Any() ? $", исторические консультанты: {_historicalConsultants.Count}" : "";
        return base.GetInfo() + hybrid + footage + consultants;
    }

    public override double CalculateRating()
    {
        double rating = base.CalculateRating();
        if (_hasDocumentaryElements) rating += 0.3;
        if (_archiveFootageMinutes > 10) rating += 0.2;
        if (_historicalConsultants.Any()) rating += 0.2;
        return Math.Min(rating, 10.0);
    }

    public string GetFilmType()
    {
        return _hasDocumentaryElements ? "Гибридный фильм" : "Художественный фильм с элементами документального";
    }

    void IAwardWinning.DisplayAwards() 
    {
        string type = _hasDocumentaryElements ? "Гибридный" : "Художественный";
        Console.WriteLine($"{type} фильм «{Title}» собрал {AwardsCount} наград");
    }
    
    public new void ReceiveAward(string awardName)
    {
        AwardsCount++;
        Console.WriteLine($"Гибридный фильм «{Title}» получил награду: {awardName}");
        AwardReceived?.Invoke(Title, awardName);
    }

    public double GetDocumentaryRatio()
    {
        if (Duration == 0) return 0;
        return (_archiveFootageMinutes / (double)Duration) * 100;
    }
    
    public void AddHistoricalConsultant(string consultantName)
    {
        if (!string.IsNullOrWhiteSpace(consultantName) && !_historicalConsultants.Contains(consultantName))
        {
            _historicalConsultants.Add(consultantName);
            Console.WriteLine($"Добавлен исторический консультант: {consultantName} для фильма «{Title}»");
        }
    }
    
    public double GetHistoricalAccuracyScore()
    {
        double score = 0;
        if (_hasDocumentaryElements) score += 30;
        if (_archiveFootageMinutes > 0) score += 30;
        if (_historicalConsultants.Any()) score += 40;
        
        return Math.Min(score, 100);
    }
}

public class ShortFilm : Movie, IAwardWinning
{
    private string _filmType;
    private int _maxDuration;
    private IAwardService _awardService;
    private List<string> _festivals; // Новая коллекция
    private string _festivalCategory; // Новый атрибут

    int IAwardWinning.AwardsCount 
    { 
        get => _awardService?.GetTotalAwards() ?? 0;
        set { }
    }
    
    public event AwardReceivedHandler AwardReceived;

    public string FilmType
    {
        get => _filmType;
        set => _filmType = !string.IsNullOrWhiteSpace(value) ? value : "Короткометражный";
    }

    public int MaxDuration
    {
        get => _maxDuration;
        set => _maxDuration = value > 0 && value <= 60 ? value : 30;
    }
    
    public string FestivalCategory
    {
        get => _festivalCategory;
        set => _festivalCategory = !string.IsNullOrWhiteSpace(value) ? value : "Общая";
    }
    
    public IReadOnlyList<string> Festivals => _festivals.AsReadOnly();

    public ShortFilm(string title, int year, string director, int duration, 
                    string filmType, IAwardService awardService)
        : base(title, year, director, duration)
    {
        FilmType = filmType;
        MaxDuration = duration <= 60 ? duration : 30;
        _awardService = awardService ?? new AwardService();
        _festivals = new List<string>(); 
        SubscribeToAwardEvents();
    }

    public ShortFilm(string title, int year, string director, int duration,
                    string filmType, string country, string language, 
                    string productionStudio, IAwardService awardService)
        : base(title, year, director, duration, country, language, productionStudio)
    {
        FilmType = filmType;
        MaxDuration = duration <= 60 ? duration : 30;
        _awardService = awardService ?? new AwardService();
        _festivals = new List<string>();
        SubscribeToAwardEvents();
    }
    
    private void SubscribeToAwardEvents()
    {
        if (_awardService is AwardService awardSvc)
        {
            awardSvc.AwardRegistered += (sender, args) =>
            {
                Console.WriteLine($"Зарегистрирована награда для «{Title}»: {args.AwardName}");
                AwardReceived?.Invoke(Title, args.AwardName);
            };
        }
    }

    void IAwardWinning.DisplayAwards()
    {
        Console.WriteLine($"Короткометражный фильм «{Title}» - история наград:");
        var awardsCount = _awardService.GetTotalAwards();
        if (awardsCount > 0)
        {
            foreach (var award in _awardService.GetAwardHistory())
            {
                Console.WriteLine($"  🏆 {award}");
            }
            Console.WriteLine($"Всего наград: {awardsCount}");
        }
        else
        {
            Console.WriteLine("  Наград пока нет");
        }
    }

    public void RegisterAward(string awardName)
    {
        _awardService.RegisterAward(awardName, DateTime.Now.Year);
    }

    public void RegisterAward(string awardName, int year)
    {
        _awardService.RegisterAward(awardName, year);
    }

    public override string GetInfo()
    {
        string festivalInfo = _festivals.Any() ? $", фестивали: {_festivals.Count}" : "";
        return $"«{Title}» ({Year}), реж. {Director}, {Duration} мин, тип: {FilmType}{festivalInfo}";
    }

    public bool IsWithinDurationLimit()
    {
        return Duration <= MaxDuration;
    }

    public string GetDurationStatus()
    {
        return IsWithinDurationLimit() 
            ? "Соответствует стандартам короткометражки" 
            : $"Превышает лимит на {Duration - MaxDuration} мин";
    }

    public override string GetTechnicalInfo()
    {
        string festivalsInfo = _festivals.Any() ? $", участвовал в фестивалях: {string.Join(", ", _festivals.Take(3))}" : "";
        return base.GetTechnicalInfo() + $", максимальная длительность: {MaxDuration} мин{festivalsInfo}";
    }
    
    public void AddFestival(string festivalName)
    {
        if (!string.IsNullOrWhiteSpace(festivalName) && !_festivals.Contains(festivalName))
        {
            _festivals.Add(festivalName);
            Console.WriteLine($"Фильм «{Title}» добавлен в фестиваль: {festivalName}");
        }
    }
    
    public string GetFestivalStatus()
    {
        if (!_festivals.Any()) return "Не участвовал в фестивалях";
        
        int internationalCount = _festivals.Count(f => 
            f.Contains("Международный") || f.Contains("International") ||
            f.Contains("Канн") || f.Contains("Венеция") || f.Contains("Берлин"));
        
        return $"Участвовал в {_festivals.Count} фестивалях " +
               $"({internationalCount} международных)";
    }
    
    public bool IsSuitableForStudentFestivals()
    {
        return Duration <= 30 && Year >= DateTime.Now.Year - 5;
    }
}


        Console.WriteLine("=== ДЕМОНСТРАЦИЯ РАСШИРЕННОЙ СИСТЕМЫ ФИЛЬМОВ ===\n");

        var awardService = new AwardService();
        var financialService = new FinancialService();

        awardService.AwardRegistered += (sender, args) =>
        {
            Console.WriteLine($"[СИСТЕМА] Зарегистрирована новая награда: " +
                            $"{args.AwardName} ({args.Year}) в {args.RegistrationTime:HH:mm:ss}");
        };

        financialService.FinancialCalculated += (sender, args) =>
        {
            Console.WriteLine($"[СИСТЕМА] Финансовые показатели для «{args.MovieTitle}»: " +
                            $"Выручка: ${args.Revenue:F1} млн, ROI: {args.ROI:F1}%");
        };

        var interstellar = new FeatureFilm("Интерстеллар", 2014, "Кристофер Нолан", 
            "фантастика", 165, 169, "США", "Английский", "Warner Bros", financialService)
        {
            BoxOffice = 677,
            MainActor = "Мэттью Макконахи"
        };
        
        var earthlings = new Documentary("Земляне", 2005, "Шон Монсон", 
            "права животных", true, 95, "США", "Английский", "Animal Planet")
        {
            Narrator = "Хоакин Феникс"
        };
        
        var avatar = new HybridFilm("Аватар", 2009, "Джеймс Кэмерон", "фантастика", 
            237, 162, true, "США", "Английский", "20th Century Fox", 15)
        {
            MainActor = "Сэм Уортингтон"
        };

        var shortFilm = new ShortFilm("Последний день лета", 2023, "Иван Петров", 
                                     25, "Экспериментальный", awardService)
        {
            FestivalCategory = "Студенческий"
        };

        var blockbuster = new FeatureFilm("Галактические войны", 2024, "Анна Сидорова", 
                                         "фантастика", 200, 148, financialService)
        {
            BoxOffice = 850,
            MainActor = "Алексей Иванов"
        };

        interstellar.RatingChanged += (title, oldRating, newRating) =>
        {
            Console.WriteLine($"[ОБНОВЛЕНИЕ] Рейтинг «{title}» изменился: {oldRating:F1} → {newRating:F1}");
        };

        interstellar.BoxOfficeMilestoneReached += (sender, args) =>
        {
            Console.WriteLine($"[ПОЗДРАВЛЕНИЕ] Фильм «{args.MovieTitle}» достиг сбора ${args.Milestone} млн!");
        };

        interstellar.AwardReceived += (title, award) =>
        {
            Console.WriteLine($"[НАГРАДА] Фильм «{title}» получил: {award}");
        };

        interstellar.MovieReleased += (movie) =>
        {
            Console.WriteLine($"[РЕЛИЗ] Фильм «{movie.Title}» теперь доступен для просмотра!");
        };

        var featureCollection = new FilmCollection<FeatureFilm>();
        featureCollection.CollectionChanged += (sender, args) =>
        {
            Console.WriteLine($"[КОЛЛЕКЦИЯ] Фильм «{args.Movie.Title}» " +
                            $"{args.ChangeType.ToString().ToLower()} в коллекции");
        };

        var documentaryCollection = new FilmCollection<Documentary>();
        var hybridCollection = new FilmCollection<HybridFilm>();
        var shortFilmCollection = new FilmCollection<ShortFilm>();

        featureCollection.AddFilm(interstellar);
        featureCollection.AddFilm(blockbuster);
        documentaryCollection.AddFilm(earthlings);
        hybridCollection.AddFilm(avatar);
        shortFilmCollection.AddFilm(shortFilm);

        interstellar.AddActors("Энн Хэтэуэй", "Джессика Честейн", "Майкл Кейн");
        interstellar.AddTags("космос", "наука", "приключения", "драма");
        
        blockbuster.AddActors("Мария Петрова", "Сергей Смирнов");
        blockbuster.AddTags("фантастика", "экшн", "война");
        
        avatar.AddActors("Зои Салдана", "Сигорни Уивер");
        avatar.AddTags("планета", "природа", "будущее");
        
        earthlings.AddActor("Пол Маккартни");
        earthlings.AddTags("документальный", "природа", "животные");

        earthlings.AddExpert("Доктор биологических наук Иванов");
        earthlings.AddExpert("Профессор экологии Петрова");
        
        avatar.AddHistoricalConsultant("Доктор исторических наук Сидоров");
        avatar.AddHistoricalConsultant("Профессор антропологии Козлова");

        shortFilm.AddFestival("Каннский кинофестиваль");
        shortFilm.AddFestival("Международный студенческий фестиваль");
        shortFilm.AddFestival("Берлинале");

        var movies = new List<Movie> { interstellar, earthlings, avatar, shortFilm, blockbuster };

        foreach (var movie in movies)
        {
            movie.Release();
        }

        Console.WriteLine("\n=== ИНФОРМАЦИЯ О ФИЛЬМАХ ===");
        foreach (var movie in movies)
        {
            Console.WriteLine(movie.GetInfo(true));
            Console.WriteLine($"Категория: {movie.GetDurationCategory()}");
            Console.WriteLine($"Возрастная категория: {movie.GetAgeCategory()}");
            Console.WriteLine($"Рейтинг: {movie.CalculateRating():F1}");
            Console.WriteLine($"Международный: {(movie.IsInternational() ? "Да" : "Нет")}");
            Console.WriteLine($"Языков: {movie.GetLanguageCount()}");
            Console.WriteLine($"Теги: {string.Join(", ", movie.Tags)}");
            Console.WriteLine();
        }

        Console.WriteLine("=== АКТЕРСКИЙ СОСТАВ ===");
        interstellar.DisplayCast();
        earthlings.DisplayCast();

        Console.WriteLine("\n=== СИСТЕМА НАГРАД ===");
        
        interstellar.ReceiveAward("Оскар за лучшие визуальные эффекты");
        interstellar.ReceiveAward("Золотой глобус за лучшую музыку");
        
        earthlings.ReceiveAward("Приз за лучший документальный фильм");
        
        avatar.ReceiveAward("Оскар за лучшую операторскую работу");
        
        shortFilm.RegisterAward("Лучший короткометражный фильм");
        shortFilm.RegisterAward("Приз зрительских симпатий");
        shortFilm.RegisterAward("Гран-при фестиваля", 2022);

        var awardMovies = new List<IAwardWinning> { interstellar, earthlings, avatar, shortFilm };
        foreach (var awardMovie in awardMovies)
        {
            awardMovie.DisplayAwards();
        }

        Console.WriteLine("\n=== ФИНАНСОВАЯ ИНФОРМАЦИЯ ===");
        Console.WriteLine(interstellar.GetFinancialReport());
        Console.WriteLine();
        Console.WriteLine(blockbuster.GetFinancialReport());

        blockbuster.BoxOffice = 105;
        blockbuster.BoxOffice = 210;
        blockbuster.BoxOffice = 850;

        Console.WriteLine("\n=== МНОГОЯЗЫЧНЫЕ ВЕРСИИ ===");
        blockbuster.AddLanguage("Английский");
        blockbuster.AddLanguage("Французский");
        blockbuster.AddLanguage("Японский");
        blockbuster.AddLanguage("Китайский");
        
        interstellar.AddLanguage("Русский");
        interstellar.AddLanguage("Испанский");

        foreach (var movie in movies)
        {
            if (movie.IsMultilingual())
            {
                Console.WriteLine($"Фильм «{movie.Title}» многоязычный: {movie.IsMultilingual()}");
                Console.WriteLine($"  Техническая информация: {movie.GetTechnicalInfo()}");
            }
        }

        Console.WriteLine("\n=== СИСТЕМА ОТЗЫВОВ ===");
        var review1 = new Review("Критик1", 9, "Отличный фильм!");
        review1.VoteHelpful();
        review1.VoteHelpful();
        review1.VoteUnhelpful();
        
        interstellar.AddReview(review1);
        interstellar.AddReview(new Review("Критик2", 8, "Великолепная операторская работа"));
        interstellar.AddReview("Зритель", 10, "Шедевр!");
        
        blockbuster.AddReview("Критик3", 7, "Хороший блокбастер");
        blockbuster.AddReview("Критик4", 6, "Средненько");

        interstellar.ShowAllReviews();
        
        Console.WriteLine("\n=== АНАЛИЗ ОТЗЫВОВ ===");
        var analysis = interstellar.AnalyzeReviews();
        analysis.Display();

        Console.WriteLine("\n=== СПЕЦИФИЧЕСКИЕ МЕТОДЫ ===");
        interstellar.ShowTrailer();
        interstellar.ShowTrailer("YouTube");
        
        earthlings.ConductInterview();
        earthlings.ConductInterview("доктор наук Иванов");

        Console.WriteLine($"Аватар - тип фильма: {avatar.GetFilmType()}");
        Console.WriteLine($"Аватар - доля архивных кадров: {avatar.GetDocumentaryRatio():F1}%");
        Console.WriteLine($"Аватар - оценка исторической достоверности: {avatar.GetHistoricalAccuracyScore():F1}%");
        Console.WriteLine($"Земляне имеет академическую поддержку: {earthlings.HasAcademicSupport()}");
        Console.WriteLine($"Земляне научно достоверен: {earthlings.IsScientificallyAccurate()}");
        Console.WriteLine($"Интерстеллар блокбастер: {interstellar.IsBlockbuster()}");
        Console.WriteLine($"Интерстеллар часть франшизы: {interstellar.IsFranchiseFilm()}");
        Console.WriteLine($"Прогноз сборов Интерстеллар: ${interstellar.ForecastFutureRevenue():F1} млн");
        Console.WriteLine($"Короткометражка соответствует стандартам: {shortFilm.IsWithinDurationLimit()}");
        Console.WriteLine($"Статус короткометражки: {shortFilm.GetDurationStatus()}");
        Console.WriteLine($"Статус фестивалей короткометражки: {shortFilm.GetFestivalStatus()}");
        Console.WriteLine($"Подходит для студенческих фестивалей: {shortFilm.IsSuitableForStudentFestivals()}");
        Console.WriteLine($"Статус исследования: {earthlings.GetResearchStatus()}");

        Console.WriteLine("\n=== УЧАСТНИКИ ДОКУМЕНТАЛЬНОГО ФИЛЬМА ===");
        foreach (var participant in earthlings.GetAllParticipants())
        {
            Console.WriteLine($"- {participant}");
        }

        Console.WriteLine("\n=== ПОИСК И РЕКОМЕНДАЦИИ ===");
        var found = featureCollection.FindFilmByTitle("Интерстеллар");
        if (found != null)
        {
            Console.WriteLine($"Найден фильм: {found.GetInfo()}");
            
            var similarByTags = interstellar.FindSimilarByTags(movies, 1);
            Console.WriteLine("\nПохожие фильмы по тегам:");
            foreach (var similar in similarByTags)
            {
                Console.WriteLine($"- {similar.Title} (общие теги: " +
                                $"{string.Join(", ", similar.Tags.Intersect(interstellar.Tags))})");
            }
        }

        Console.WriteLine("\n=== СТАТИСТИКА КОЛЛЕКЦИЙ ===");
        Console.WriteLine("Художественные фильмы:");
        featureCollection.GetStatistics().Display();
        
        Console.WriteLine("\n=== ФИЛЬМЫ С ВЫСОКИМ РЕЙТИНГОМ ===");
        var highRatedFilms = featureCollection.GetFilmsWithRatingAbove(7.5);
        foreach (var film in highRatedFilms)
        {
            Console.WriteLine($"- {film.Title}: {film.CalculateRating():F1}");
        }

        Console.WriteLine("\n=== ВСЕ КОЛЛЕКЦИИ ===");
        featureCollection.DisplayAllFilms();
        documentaryCollection.DisplayAllFilms();
        hybridCollection.DisplayAllFilms();
        shortFilmCollection.DisplayAllFilms();

        Console.WriteLine("\n=== ДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ ===");
        foreach (var movie in movies)
        {
            Console.WriteLine($"Фильм: {movie.Title}");
            Console.WriteLine($"  Детальная информация: {movie.GetInfo(true)}");
            if (movie is FeatureFilm feature)
            {
                Console.WriteLine($"  Финансовый ROI: {feature.CalculateROI():F1}%");
                if (feature.Screenwriters.Any())
                    Console.WriteLine($"  Сценаристы: {string.Join(", ", feature.Screenwriters)}");
            }
            if (movie is Documentary doc)
            {
                Console.WriteLine($"  Образовательный: {doc.IsEducational}");
                Console.WriteLine($"  Экспертов: {doc.Experts.Count}");
            }
            if (movie is HybridFilm hybrid)
            {
                Console.WriteLine($"  Консультантов: {hybrid.HistoricalConsultants.Count}");
            }
            if (movie is ShortFilm shortF)
            {
                Console.WriteLine($"  Категория фестиваля: {shortF.FestivalCategory}");
            }
            Console.WriteLine();
        }

        interstellar.MarkAsViewed();
        blockbuster.MarkAsViewed();

        Console.WriteLine("=== ДЕМОНСТРАЦИЯ ЗАВЕРШЕНА ===");


=== ДЕМОНСТРАЦИЯ РАСШИРЕННОЙ СИСТЕМЫ ФИЛЬМОВ ===

Добавлен актер: Мэттью Макконахи в фильм «Интерстеллар»
Добавлен актер: Хоакин Феникс в фильм «Земляне»
Добавлен актер: Сэм Уортингтон в фильм «Аватар»
Добавлен актер: Алексей Иванов в фильм «Галактические войны»
Фильм «Интерстеллар» добавлен в коллекцию
[КОЛЛЕКЦИЯ] Фильм «Интерстеллар» added в коллекции
Фильм «Галактические войны» добавлен в коллекцию
[КОЛЛЕКЦИЯ] Фильм «Галактические войны» added в коллекции
Фильм «Земляне» добавлен в коллекцию
Фильм «Аватар» добавлен в коллекцию
Фильм «Последний день лета» добавлен в коллекцию
Добавлен актер: Энн Хэтэуэй в фильм «Интерстеллар»
Добавлен актер: Джессика Честейн в фильм «Интерстеллар»
Добавлен актер: Майкл Кейн в фильм «Интерстеллар»
Добавлен тег: космос к фильму «Интерстеллар»
Добавлен тег: наука к фильму «Интерстеллар»
Добавлен тег: приключения к фильму «Интерстеллар»
Добавлен тег: драма к фильму «Интерстеллар»
Добавлен актер: Мария Петрова в фильм «Галактические войны»
Добавлен актер: